Григорьева Анастасия 3391 (Домашнее задание №4)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv('jamb_exam_results.csv') #Загрузка данных

df.columns = df.columns.str.lower().str.replace(' ', '_') #Преобразование названий колонок

df = df.drop('student_id', axis=1)
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

#Разделение на признаки и целевую переменную
y_train = df_train.jamb_score.values
y_val = df_val.jamb_score.values
y_test = df_test.jamb_score.values

del df_train['jamb_score']
del df_val['jamb_score']
del df_test['jamb_score']

#Преобразование в матрицы
train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

#Вопрос 1
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
tree_feature = feature_names[dt.tree_.feature[0]]
print(f"Вопрос 1: Признак для разбиения - {tree_feature}")

#Вопрос 2
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Вопрос 2: RMSE = {rmse:.2f}")

#Вопрос 3
scores = []
for n in range(10, 201, 10):
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    scores.append((n, rmse))

df_scores = pd.DataFrame(scores, columns=['n_estimators', 'rmse'])
min_rmse = df_scores.loc[df_scores['rmse'].idxmin()]
print(f"Вопрос 3: Лучшее n_estimators = {min_rmse['n_estimators']}")

#Вопрос 4
scores = []
for depth in [10, 15, 20, 25]:
    for n in range(10, 201, 10):
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        scores.append((depth, n, rmse))

df_scores = pd.DataFrame(scores, columns=['max_depth', 'n_estimators', 'rmse'])
mean_rmse = df_scores.groupby('max_depth')['rmse'].mean()
best_depth = mean_rmse.idxmin()
print(f"Вопрос 4: Лучший max_depth = {best_depth}")

#Вопрос 5
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train, y_train)

feature_importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
})
top_feature = feature_importances.sort_values('importance', ascending=False).iloc[0]
print(f"Вопрос 5: Самый важный признак - {top_feature['feature']}")

ОТВЕТЫ НА ВОПРОСЫ:

1. Вопрос: study_hours_per_week
2. Вопрос: 42.13
3. Вопрос: 80
4. Вопрос: 10
5. Вопрос: study_hours_per_week